In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
import os
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Path to dataset root
data_dir = "/kaggle/input/datasets/sahityapettugani/plant-village/color"

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load full dataset
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Create indices
indices = list(range(len(full_dataset)))

# First split: train (80%) and temp (20%)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, stratify=full_dataset.targets, random_state=42)

# Second split: val (10%) and test (10%)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[full_dataset.targets[i] for i in temp_idx], random_state=42)

# Create subsets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")


Train size: 43441
Val size: 5430
Test size: 5431


In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
MODEL_PATH   = "/kaggle/input/models/a0282747r/simple-cnn/pytorch/default/1"
DATASET_PATH = "/kaggle/input/datasets/sahityapettugani/plant-village"
OUTPUT_DIR   = "/kaggle/working/results_grid"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# HYPERPARAMETERS
# ─────────────────────────────────────────────
OPTIMIZERS    = ["adam", "sgd"]
LEARNING_RATES = [0.1]
EPOCHS        = 15
BATCH_SIZE    = 32
IMG_SIZE      = 224
NUM_WORKERS   = 2
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ─────────────────────────────────────────────
# DATA TRANSFORMS
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ─────────────────────────────────────────────
# DATASET — auto-detect subfolder structure
# ─────────────────────────────────────────────
# PlantVillage sometimes has a nested folder (e.g. /plant-village/PlantVillage/...)
# This finds the right root automatically.
def find_image_root(base_path):
    for root, dirs, files in os.walk(base_path):
        # A valid image folder root has multiple class subdirectories
        subdirs = [d for d in dirs if not d.startswith('.')]
        if len(subdirs) >= 5:
            return root
    return base_path

image_root = find_image_root(DATASET_PATH)
print(f"Image root detected: {image_root}")

full_dataset = datasets.ImageFolder(image_root, transform=train_transform)
NUM_CLASSES  = len(full_dataset.classes)
print(f"Classes found: {NUM_CLASSES}")
print(f"Total images:  {len(full_dataset)}")

# Train / val / test split
total       = len(full_dataset)
test_size   = int(total * TEST_SPLIT)
val_size    = int(total * VAL_SPLIT)
train_size  = total - val_size - test_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
# Apply val/test transform (no augmentation)
val_ds.dataset.transform  = val_transform
test_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────
def load_model(model_path, num_classes):
    """
    Tries to load the saved model. Handles:
      1. A full model saved with torch.save(model, ...)  → torch.load
      2. A state_dict saved with torch.save(model.state_dict(), ...)
         → requires architecture to be defined separately
    """
    model_files = [f for f in os.listdir(model_path)
                   if f.endswith(('.pt', '.pth', '.bin'))]
    if not model_files:
        raise FileNotFoundError(f"No .pt/.pth/.bin file found in {model_path}")

    model_file = os.path.join(model_path, model_files[0])
    print(f"Loading model from: {model_file}")

    obj = torch.load(model_file, map_location=DEVICE)

    # Case 1: full model object
    if isinstance(obj, nn.Module):
        model = obj
        # Replace final classifier layer to match num_classes
        _replace_head(model, num_classes)
        return model

    # Case 2: state dict — define a simple CNN backbone to load into
    if isinstance(obj, dict):
        print("Detected state_dict — building SimpleCNN backbone")
        model = SimpleCNN(num_classes)
        try:
            model.load_state_dict(obj, strict=False)
        except Exception as e:
            print(f"Warning: partial load — {e}")
        return model

    raise ValueError("Unrecognised model file format.")


def _replace_head(model, num_classes):
    """Replace the last Linear layer with one matching num_classes."""
    for name in ['fc', 'classifier', 'head', 'linear']:
        layer = getattr(model, name, None)
        if layer is not None:
            if isinstance(layer, nn.Linear):
                setattr(model, name, nn.Linear(layer.in_features, num_classes))
                print(f"Replaced '{name}' head → {num_classes} classes")
                return
            if isinstance(layer, nn.Sequential):
                for i in reversed(range(len(layer))):
                    if isinstance(layer[i], nn.Linear):
                        layer[i] = nn.Linear(layer[i].in_features, num_classes)
                        print(f"Replaced '{name}[{i}]' → {num_classes} classes")
                        return


class SimpleCNN(nn.Module):
    """Fallback architecture if only a state_dict is found."""
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# ─────────────────────────────────────────────
# TRAIN / EVAL HELPERS
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return running_loss / total, correct / total


# ─────────────────────────────────────────────
# PLOT HELPER
# ─────────────────────────────────────────────
def save_curves(history, run_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs_range = range(1, len(history['train_acc']) + 1)

    ax1.plot(epochs_range, history['train_acc'], label='Train', color='#3266ad', linewidth=2)
    ax1.plot(epochs_range, history['val_acc'],   label='Val',   color='#3266ad', linewidth=1.5, linestyle='--')
    ax1.set_title(f'{run_name} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history['train_loss'], label='Train', color='#e2783c', linewidth=2)
    ax2.plot(epochs_range, history['val_loss'],   label='Val',   color='#e2783c', linewidth=1.5, linestyle='--')
    ax2.set_title(f'{run_name} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, f"{run_name}_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"Saved plot: {path}")


# ─────────────────────────────────────────────
# GRID TRAINING LOOP
# ─────────────────────────────────────────────
criterion  = nn.CrossEntropyLoss()
summary    = []

for opt_name, lr in itertools.product(OPTIMIZERS, LEARNING_RATES):
    run_name = f"{opt_name}_lr_{str(lr).replace('.', '_')}"
    print(f"\n{'='*60}")
    print(f"Starting run: {run_name}")
    print(f"{'='*60}")

    model = load_model(MODEL_PATH, NUM_CLASSES).to(DEVICE)

    if opt_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    best_val_acc = 0.0

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_path = os.path.join(OUTPUT_DIR, f"{run_name}_best.pth")
            torch.save(model.state_dict(), best_path)

        print(f"Epoch {epoch:02d}/{EPOCHS} | "
              f"Train acc: {tr_acc:.4f} loss: {tr_loss:.4f} | "
              f"Val acc: {vl_acc:.4f} loss: {vl_loss:.4f}")

    # Test evaluation
    print(f"\nEvaluating {run_name} on test set...")
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f"Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")

    save_curves(history, run_name)

    # Save final model
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"{run_name}_final.pth"))

    summary.append({
        'run':          run_name,
        'optimizer':    opt_name,
        'lr':           lr,
        'test_acc':     round(test_acc, 4),
        'test_loss':    round(test_loss, 4),
        'best_val_acc': round(best_val_acc, 4),
    })

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
df = pd.DataFrame(summary).sort_values('test_acc', ascending=False)
csv_path = os.path.join(OUTPUT_DIR, "summary_results.csv")
df.to_csv(csv_path, index=False)
print(f"\nSummary CSV saved to: {csv_path}")
print("\nRanked results:")
for _, row in df.iterrows():
    print(f"  {row['run']} | test_acc={row['test_acc']} | "
          f"test_loss={row['test_loss']} | best_val_acc={row['best_val_acc']}")

Using device: cuda
Image root detected: /kaggle/input/datasets/sahityapettugani/plant-village/color
Classes found: 38
Total images:  54302
Train: 40727 | Val: 8145 | Test: 5430

Starting run: adam_lr_0_1
Loading model from: /kaggle/input/models/a0282747r/simple-cnn/pytorch/default/1/simple_net.pth
Detected state_dict — building SimpleCNN backbone
Epoch 01/15 | Train acc: 0.0992 loss: 1260.2488 | Val acc: 0.0911 loss: 3.3693
Epoch 02/15 | Train acc: 0.0965 loss: 3.3688 | Val acc: 0.0911 loss: 3.3725
Epoch 03/15 | Train acc: 0.1000 loss: 3.3694 | Val acc: 0.1057 loss: 3.3568
Epoch 04/15 | Train acc: 0.0975 loss: 3.3705 | Val acc: 0.0911 loss: 3.3802
Epoch 05/15 | Train acc: 0.0981 loss: 3.3708 | Val acc: 0.0911 loss: 3.3582
Epoch 06/15 | Train acc: 0.0972 loss: 3.3576 | Val acc: 0.1057 loss: 3.3490
Epoch 07/15 | Train acc: 0.0990 loss: 3.3579 | Val acc: 0.0911 loss: 3.3519
Epoch 08/15 | Train acc: 0.0988 loss: 3.3583 | Val acc: 0.1057 loss: 3.3473
Epoch 09/15 | Train acc: 0.0987 loss: 3.